In [2]:
!pip install pandas numpy scipy


  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.9 MB 14.9 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.9 MB 14.6 MB/s eta 0:00:01
   --------------------------- ------------ 6.8/9.9 MB 12.4 MB/s eta 0:00:01
   ----------------------------------- ---- 8.7/9.9 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 9.9 MB/s  0:00:01
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.4 MB 5.2 MB/s eta 0:00:03
   ------ --------------------------------- 2.1/12.4 MB 5.5 MB/s eta 0:00:02
   ---------- ----------------------------- 3.4/12.4 MB 5.6 MB/s eta 0:00:02
   -------------- ------------------------- 4.5/12.4 MB 5.5 MB/s eta 0:00:02
   ------------------- -------------------- 6.0/12.4 MB 5.7 MB/s eta 0:00:02
   --------------------- --

In [4]:
import pandas as pd

df = pd.read_csv("../data/processed/user_experiment_summary.csv")
df.head()


,experiment_id,user_id,variant,assigned_at,segment,country,channel,did_convert,orders_count,total_revenue,avg_order_value,refund_count,ticket_count
0,pricing_discount_v1,1,treatment,2024-01-07,SMB,IN,Referral,0,0,0.0,NaN,0,0
1,pricing_discount_v1,2,control,2024-01-06,SMB,UK,Organic,1,1,49.0,49.0,0,0
2,pricing_discount_v1,3,control,2024-01-06,SMB,US,Direct,0,0,0.0,NaN,0,0
3,pricing_discount_v1,4,control,2024-01-02,SMB,IN,Organic,0,0,0.0,NaN,0,0
4,pricing_discount_v1,5,treatment,2024-01-03,SMB,UK,Paid Search,0,0,0.0,NaN,0,0


In [5]:
df.shape

(50000, 13)

In [6]:
df["variant"].value_counts()


variant
control      25020
treatment    24980
Name: count, dtype: int64

In [7]:
kpi_summary = (
    df.groupby("variant")
      .agg(
          users=("user_id", "count"),
          conversion_rate=("did_convert", "mean"),
          revenue_per_user=("total_revenue", "mean"),
          avg_order_value=("avg_order_value", "mean"),
          tickets_per_user=("ticket_count", "mean")
      )
)

kpi_summary


,users,conversion_rate,revenue_per_user,avg_order_value,tickets_per_user
variant,,,,,
control,25020,0.138209,20.581215,148.913245,0.098761
treatment,24980,0.170777,22.890336,134.036709,0.100961


In [8]:
control = kpi_summary.loc["control"]
treatment = kpi_summary.loc["treatment"]

uplift = pd.DataFrame({
    "metric": kpi_summary.columns,
    "control": control.values,
    "treatment": treatment.values
})

uplift["absolute_change"] = uplift["treatment"] - uplift["control"]
uplift["percent_change"] = (uplift["absolute_change"] / uplift["control"]) * 100

uplift


,metric,control,treatment,absolute_change,percent_change
0,users,25020.000000,24980.000000,-40.000000,-0.159872
1,conversion_rate,0.138209,0.170777,0.032567,23.563651
2,revenue_per_user,20.581215,22.890336,2.309121,11.219557
3,avg_order_value,148.913245,134.036709,-14.876536,-9.990069
4,tickets_per_user,0.098761,0.100961,0.002200,2.227375


In [9]:
from scipy import stats

# Conversion test (binary metric)
conv_control = df[df.variant == "control"]["did_convert"]
conv_treatment = df[df.variant == "treatment"]["did_convert"]

t_stat, p_conv = stats.ttest_ind(conv_treatment, conv_control, equal_var=False)

print("Conversion p-value:", p_conv)

# Revenue per user test
rev_control = df[df.variant == "control"]["total_revenue"]
rev_treatment = df[df.variant == "treatment"]["total_revenue"]

t_stat, p_rev = stats.ttest_ind(rev_treatment, rev_control, equal_var=False)

print("Revenue per user p-value:", p_rev)


Conversion p-value: 6.84563805205619e-24
Revenue per user p-value: 6.804937620886752e-05


In [10]:
segment_summary = (
    df.groupby(["segment", "variant"])
      .agg(
          users=("user_id", "count"),
          conversion_rate=("did_convert", "mean"),
          revenue_per_user=("total_revenue", "mean")
      )
      .reset_index()
)

segment_summary


,segment,variant,users,conversion_rate,revenue_per_user
0,Enterprise,control,2513,0.075209,11.385197
1,Enterprise,treatment,2503,0.085497,11.662964
2,Mid-Market,control,7524,0.126528,18.998937
3,Mid-Market,treatment,7558,0.157052,21.036875
4,SMB,control,14983,0.154642,22.918174
5,SMB,treatment,14919,0.192037,25.712950


In [11]:
segment_pivot = segment_summary.pivot(
    index="segment",
    columns="variant",
    values=["conversion_rate", "revenue_per_user"]
)

segment_pivot


conversion_rate           revenue_per_user           
variant            control treatment          control  treatment
segment                                                         
Enterprise        0.075209  0.085497        11.385197  11.662964
Mid-Market        0.126528  0.157052        18.998937  21.036875
SMB               0.154642  0.192037        22.918174  25.712950

Recommendation:
The pricing discount experiment led to a statistically significant increase in conversion rate and revenue per user overall.
Segment-level analysis shows the strongest uplift among SMB and Mid-Market customers, while Enterprise users show minimal response.

Proposed action:
Roll out the discount strategy to SMB and Mid-Market segments, and run a separate targeted experiment for Enterprise customers to test alternative incentives.